# PG-Stack · Step 1 — Data, Patient-Grouped Folds & Tabular Base Learners

**Project:** Patient-Graph Stacking (PG-Stack) for 3D total-body skin cancer triage (ISIC 2024 / SLICE-3D).

This notebook:
1. Loads the SLICE-3D metadata and implements the **official metric** (pAUC above 80% TPR).
2. Creates **patient-grouped, stratified 5-fold CV** → `folds.csv`. *Every later notebook reuses these folds* (no leakage between base learners and the meta-learner).
3. Engineers per-lesion features.
4. Trains two LightGBM models and saves out-of-fold (OOF) predictions:
   - **`tab_lesion`** – lesion-only features → a *base learner* for stacking (deliberately has **no** patient context; the graph meta-learner must learn it).
   - **`tab_pnorm`** – lesion + hand-crafted patient-normalised (“ugly duckling”) features, ISIC-2024-winner style → the **strong baseline to beat**.

**Kaggle setup:** *Add Input → Competitions → `isic-2024-challenge`* (join the competition and accept the rules first). Accelerator: none needed (CPU is fine). Internet: not needed.

In [ ]:
import os, gc, json, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_curve, auc, roc_auc_score
warnings.filterwarnings("ignore")

# ---------------- Config ----------------
DATA_DIR = "/kaggle/input/isic-2024-challenge"
OUT_DIR  = "/kaggle/working"
SEED     = 42
N_FOLDS  = 5
NEG_SAMPLE_RATIO = 0.01   # keep 1% of negatives per bag (winners' trick: fast + strong)
N_BAGS   = 5              # bagged seeds per fold
USE_ATTRIBUTION = False   # hospital ID = possible shortcut; keep False for a fair/generalisable study

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)
print("lightgbm", lgb.__version__)

## 1. Load metadata

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/train-metadata.csv", low_memory=False)
print(df.shape)
print("positives:", int(df.target.sum()), f"({df.target.mean()*100:.3f}%)")
print("patients :", df.patient_id.nunique())
df.head(3)

### Columns that would leak the label (train-only / post-biopsy) — never use as features

In [ ]:
LEAK_COLS = ["lesion_id", "iddx_full", "iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5",
             "mel_mitotic_index", "mel_thick_mm", "tbp_lv_dnn_lesion_confidence"]
ID_COLS   = ["isic_id", "patient_id", "target", "image_type", "copyright_license", "attribution"]
print([c for c in LEAK_COLS if c in df.columns])

## 2. Official metric — partial AUC above 80% TPR (max score 0.2)

In [ ]:
def pauc_80(y_true, y_pred, min_tpr=0.80):
    """Official ISIC 2024 scorer (pAUC above min_tpr). Range [0, 0.2]."""
    v_gt = np.abs(np.asarray(y_true) - 1)
    v_pred = -1.0 * np.asarray(y_pred)
    max_fpr = abs(1 - min_tpr)
    fpr, tpr, _ = roc_curve(v_gt, v_pred)
    stop = np.searchsorted(fpr, max_fpr, "right")
    x_interp = [fpr[stop - 1], fpr[stop]]
    y_interp = [tpr[stop - 1], tpr[stop]]
    tpr = np.append(tpr[:stop], np.interp(max_fpr, x_interp, y_interp))
    fpr = np.append(fpr[:stop], max_fpr)
    return auc(fpr, tpr)

def report(name, y, p):
    s = dict(model=name, pauc80=pauc_80(y, p), auroc=roc_auc_score(y, p))
    # sensitivity at 95% specificity (clinically readable)
    neg = np.sort(p[y == 0]); thr = neg[int(0.95 * len(neg))]
    s["sens@spec95"] = float((p[y == 1] > thr).mean())
    return s

# sanity check: random ≈ 0.02, perfect = 0.2
_y = np.r_[np.zeros(1000), np.ones(10)]
print(round(pauc_80(_y, np.random.rand(len(_y))), 4), round(pauc_80(_y, _y), 4))

## 3. Quick EDA — why patient context matters

In [ ]:
per_pat = df.groupby("patient_id").agg(n_lesions=("isic_id", "size"), n_pos=("target", "sum"))
print(per_pat.n_lesions.describe().round(1))
print("patients with ≥1 malignant lesion:", int((per_pat.n_pos > 0).sum()))
print("\nmalignancy rate by site (%):")
print((df.groupby("anatom_site_general").target.mean() * 100).round(3).sort_values(ascending=False))

## 4. Patient-grouped, stratified folds
All lesions of a patient go to the same fold — essential because the meta-learner will build **per-patient graphs**.

In [ ]:
pat = per_pat.reset_index()
pat["has_pos"] = (pat.n_pos > 0).astype(int)
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
df["fold"] = -1
for f, (_, va) in enumerate(sgkf.split(df, df.target, groups=df.patient_id)):
    df.loc[df.index[va], "fold"] = f

assert df.groupby("patient_id").fold.nunique().max() == 1, "patient leakage!"
print(df.groupby("fold").agg(n=("target", "size"), pos=("target", "sum"),
                             patients=("patient_id", "nunique")))
df[["isic_id", "patient_id", "fold", "target"]].to_csv(f"{OUT_DIR}/folds.csv", index=False)

## 5. Feature engineering
**Lesion features** use only the lesion itself (colour, shape, border, size, 3D position).
**Patient-normalised features** (baseline only) compare each lesion to the same patient's other lesions — the hand-crafted ugly-duckling sign.

In [ ]:
NUM_RAW = [c for c in df.columns
           if (c.startswith("tbp_lv_") or c in ["age_approx", "clin_size_long_diam_mm"])
           and c not in LEAK_COLS and pd.api.types.is_numeric_dtype(df[c])]
CAT_COLS = ["sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location", "tbp_lv_location_simple"]
if USE_ATTRIBUTION: CAT_COLS.append("attribution")

def add_lesion_features(d):
    d = d.copy()
    eps = 1e-6
    d["age_approx"] = pd.to_numeric(d["age_approx"], errors="coerce")
    d["lesion_size_ratio"]        = d.tbp_lv_minorAxisMM / (d.clin_size_long_diam_mm + eps)
    d["lesion_shape_index"]       = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM ** 2 + eps)
    d["hue_contrast"]             = (d.tbp_lv_H - d.tbp_lv_Hext).abs()
    d["luminance_contrast"]       = (d.tbp_lv_L - d.tbp_lv_Lext).abs()
    d["lesion_color_difference"]  = np.sqrt(d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2)
    d["border_complexity"]        = d.tbp_lv_norm_border + d.tbp_lv_symm_2axis
    d["color_uniformity"]         = d.tbp_lv_color_std_mean / (d.tbp_lv_radial_color_std_max + eps)
    d["position_distance_3d"]     = np.sqrt(d.tbp_lv_x**2 + d.tbp_lv_y**2 + d.tbp_lv_z**2)
    d["perimeter_to_area_ratio"]  = d.tbp_lv_perimeterMM / (d.tbp_lv_areaMM2 + eps)
    d["area_to_perimeter_ratio"]  = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM + eps)
    d["lesion_visibility_score"]  = d.tbp_lv_deltaLBnorm + d.tbp_lv_norm_color
    d["symmetry_border_consistency"] = d.tbp_lv_symm_2axis * d.tbp_lv_norm_border
    d["color_shape_composite"]    = (d.tbp_lv_color_std_mean + d.tbp_lv_area_perim_ratio + d.tbp_lv_symm_2axis) / 3
    d["log_area"]                 = np.log1p(d.tbp_lv_areaMM2)
    d["normalized_lesion_size"]   = d.clin_size_long_diam_mm / (d.age_approx + eps)
    d["mean_hue_difference"]      = (d.tbp_lv_H + d.tbp_lv_Hext) / 2
    d["std_dev_contrast"]         = np.sqrt((d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2) / 3)
    return d

ENG = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast", "luminance_contrast",
       "lesion_color_difference", "border_complexity", "color_uniformity", "position_distance_3d",
       "perimeter_to_area_ratio", "area_to_perimeter_ratio", "lesion_visibility_score",
       "symmetry_border_consistency", "color_shape_composite", "log_area", "normalized_lesion_size",
       "mean_hue_difference", "std_dev_contrast"]

df = add_lesion_features(df)
LESION_NUM = sorted(set(NUM_RAW + ENG))
for c in CAT_COLS:
    df[c] = df[c].fillna("NA").astype("category")

# ---- patient-normalised (hand-crafted ugly duckling) — BASELINE ONLY ----
g = df.groupby("patient_id")
PNORM = []
for c in LESION_NUM:
    m, s = g[c].transform("mean"), g[c].transform("std")
    df[f"{c}_pnorm"] = (df[c] - m) / (s + 1e-6)
    PNORM.append(f"{c}_pnorm")
df["patient_n_lesions"] = g["isic_id"].transform("size")
PNORM.append("patient_n_lesions")

FEATS_LESION = LESION_NUM + CAT_COLS
FEATS_PNORM  = FEATS_LESION + PNORM
print(len(FEATS_LESION), "lesion features |", len(FEATS_PNORM), "with patient-norm")

## 6. LightGBM with negative under-sampling + bagging (OOF)

In [ ]:
LGB_PARAMS = dict(objective="binary", learning_rate=0.03, num_leaves=63, max_depth=7,
                  min_child_samples=40, feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1,
                  lambda_l1=0.1, lambda_l2=1.0, n_estimators=300, verbosity=-1)

def train_oof(feats, name):
    oof = np.zeros(len(df))
    imps = []
    for f in range(N_FOLDS):
        tr, va = df[df.fold != f], df[df.fold == f]
        pred = np.zeros(len(va))
        for b in range(N_BAGS):
            rs = SEED + 100 * f + b
            neg = tr[tr.target == 0].sample(frac=NEG_SAMPLE_RATIO, random_state=rs)
            sub = pd.concat([tr[tr.target == 1], neg])
            m = lgb.LGBMClassifier(**LGB_PARAMS, random_state=rs)
            m.fit(sub[feats], sub.target)
            pred += m.predict_proba(va[feats])[:, 1] / N_BAGS
            imps.append(pd.Series(m.feature_importances_, index=feats))
        oof[np.where(df.fold == f)[0]] = pred
        print(f"  [{name}] fold {f}: pAUC={pauc_80(va.target, pred):.4f}")
    imp = pd.concat(imps, axis=1).mean(axis=1).sort_values(ascending=False)
    return oof, imp

results = []
for name, feats in [("tab_lesion", FEATS_LESION), ("tab_pnorm", FEATS_PNORM)]:
    oof, imp = train_oof(feats, name)
    df[f"oof_{name}"] = oof
    results.append(report(name, df.target.values, oof))
    print(imp.head(10), "\n")

res = pd.DataFrame(results).set_index("model").round(4)
res

## 7. Save outputs for the next steps
After this finishes: **Save Version → Save & Run All**. Then in later notebooks, *Add Input → Notebook output* of this notebook.

In [ ]:
keep = ["isic_id", "patient_id", "fold", "target", "oof_tab_lesion", "oof_tab_pnorm"]
df[keep].to_csv(f"{OUT_DIR}/oof_tabular.csv", index=False)

# Lesion-level node features for the graph meta-learner (Step 4)
node = df[["isic_id", "patient_id", "fold", "target",
           "tbp_lv_x", "tbp_lv_y", "tbp_lv_z", "anatom_site_general", "tbp_lv_location_simple",
           "age_approx", "sex"] + [c for c in LESION_NUM if c not in ["tbp_lv_x", "tbp_lv_y", "tbp_lv_z", "age_approx"]]]
node.to_parquet(f"{OUT_DIR}/lesion_features.parquet", index=False)

res.to_csv(f"{OUT_DIR}/results_step1.csv")
json.dump(dict(lesion_num=LESION_NUM, cat=CAT_COLS), open(f"{OUT_DIR}/feature_lists.json", "w"))
print(os.listdir(OUT_DIR))